# Chapter 8, Part 2: cations, anions, and radicals

## Learning objectives

After this notebook, you should be able to:

1. Specify charge, electron count, and spin multiplicity separately.
2. Run small closed-shell and open-shell calculations and check their spin properties.
3. Write balanced ionization, electron-attachment, proton-loss, and bond-cleavage reactions.
4. Distinguish vertical electronic energy differences from adiabatic energies, bond dissociation enthalpies, and solution acidity.

**Prerequisites:** [Part 1](Chapter08_Part1.ipynb) and [Chapter 5](Chapter05.ipynb). Use the activated [course environment](Readme.md#set-up-python). This notebook starts from embedded coordinates and does not use files from another notebook.

**Runtime:** five tiny HF/6-31+G* single points (OH⁺, OH, OH⁻, water, H), followed by matrix operations and plots. There are no optimizations, large substituent libraries, or optional long calculations. Each single point is timed. Outputs go to `outputs/chapter08_part2/`.

### Start here: electrons, charge, and fragments

A **cation** has lost electrons relative to its neutral composition; an **anion** has gained them. A **radical** has an unpaired electron. Charge and radical character are different properties: a species can be charged and open-shell. Alpha and beta label spin projections, not two kinds of electron. The multiplicity $2S+1$ describes the total spin state chosen for a calculation.

When a bond breaks, ask where its electrons go. In *homolysis*, each fragment receives one electron from the bond; in *heterolysis*, both go to one fragment. The resulting fragments have different energies and often different charges. A reaction energy must name those fragments and their states.

**Research question:** can our fragment calculations tell a consistent energy-accounting story, and can a consistent story still be inaccurate? Follow the electron counts, the two dissociation choices, and the energy cycle. The overlap-matrix derivation of spin diagnostics is a deeper numerical check.

## 8.2.1 Charge does not determine spin

For an all-electron calculation, $N=\sum_A Z_A-Q$, where positive $Q$ means electrons have been removed. Spin multiplicity is $M=2S+1$. For the highest-$M_S$ component normally used in an SCF calculation,

$$N_\alpha=\frac{N+M-1}{2},\qquad N_\beta=\frac{N-M+1}{2}.$$

Both counts must be nonnegative integers. This condition permits a state; it does **not** prove that state is lowest in energy. Even-electron species can be open-shell, and ions can be radicals.

| Species | Charge | Electrons | Chosen multiplicity | SCF reference |
| --- | ---: | ---: | ---: | --- |
| OH⁺ | +1 | 8 | 3 (triplet) | UHF |
| OH | 0 | 9 | 2 (doublet) | UHF |
| OH⁻ | −1 | 10 | 1 (singlet) | RHF |
| H₂O | 0 | 10 | 1 (singlet) | RHF |
| H | 0 | 1 | 2 (doublet) | UHF |

RHF uses paired spatial orbitals. UHF allows different alpha and beta spatial orbitals; it can describe these open shells but may mix total-spin states. ROHF is another open-shell choice. State selection and SCF convergence must be checked independently. See [Psi4's reference and charge/multiplicity documentation](https://psicode.org/psi4manual/master/scf.html) and [Pople and Nesbet's unrestricted-orbital formulation](https://doi.org/10.1063/1.1740120).

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import numpy as np
import matplotlib.pyplot as plt
import psi4

OUT = Path("outputs/chapter08_part2")
OUT.mkdir(parents=True, exist_ok=True)
psi4.core.clean_options()
psi4.core.clean_variables()
SCRATCH = (OUT / "scratch").resolve()
SCRATCH.mkdir(parents=True, exist_ok=True)
psi4.core.IOManager.shared_object().set_default_path(str(SCRATCH))
psi4.set_num_threads(1)
psi4.set_memory("512 MiB")
psi4.core.set_output_file(str(OUT / "psi4.log"), False)
BASIS = "6-31+g*"  # diffuse heavy-atom functions are relevant to OH−
HARTREE_TO_EV = psi4.constants.hartree2ev
HARTREE_TO_KJ_MOL = psi4.constants.hartree2kJmol
print(f"Psi4 {psi4.__version__}; one thread; 512 MiB")

def spin_counts(electrons, multiplicity):
    if not isinstance(electrons, int) or not isinstance(multiplicity, int):
        raise ValueError("Electron count and multiplicity must be integers.")
    unpaired_excess = multiplicity - 1
    if electrons < 0 or unpaired_excess < 0 or unpaired_excess > electrons or (electrons + unpaired_excess) % 2:
        raise ValueError("Incompatible electron count and multiplicity.")
    return (electrons + unpaired_excess)//2, (electrons - unpaired_excess)//2

assert spin_counts(9, 2) == (5, 4)
try:
    spin_counts(9, 1)
except ValueError:
    print("Self-check: an odd-electron singlet input is rejected.")
else:
    raise AssertionError("The invalid state was not rejected.")

## 8.2.2 Small, reproducible single points

All three OH charge states use **exactly the same** O–H distance, 0.970 Å. Water contains that same O–H fragment, with a second 0.970 Å bond at 104.5°. These are specified teaching geometries, not optimized minima. Hydrogen atoms and protons introduced later are infinitely separated fragments, not additional nuclei interacting at the same coordinates.

The `+` in 6-31+G* adds diffuse heavy-atom functions, allowing a more extended electron density for anions. The `*` adds polarization functions on heavy atoms. These improvements do not establish basis convergence or repair HF's missing electron correlation. We reset options for each call so a prior PCM or DFT calculation cannot change the problem silently.

In [ ]:
bond = 0.970
angle = np.deg2rad(104.5)
oh_coordinates = np.array([[0., 0., 0.], [0., 0., bond]])
water_coordinates = np.vstack([oh_coordinates, [bond*np.sin(angle), 0, bond*np.cos(angle)]])
specifications = {
    "OH+": {"symbols": ["O", "H"], "coordinates": oh_coordinates, "charge": 1, "multiplicity": 3},
    "OH":  {"symbols": ["O", "H"], "coordinates": oh_coordinates, "charge": 0, "multiplicity": 2},
    "OH-": {"symbols": ["O", "H"], "coordinates": oh_coordinates, "charge": -1, "multiplicity": 1},
    "H2O": {"symbols": ["O", "H", "H"], "coordinates": water_coordinates, "charge": 0, "multiplicity": 1},
    "H":   {"symbols": ["H"], "coordinates": np.zeros((1, 3)), "charge": 0, "multiplicity": 2},
}
atomic_numbers = {"H": 1, "O": 8}

def single_point(specification):
    q, mult = specification["charge"], specification["multiplicity"]
    electrons = sum(atomic_numbers[s] for s in specification["symbols"]) - q
    expected_counts = spin_counts(electrons, mult)
    rows = [f"{q} {mult}"]
    rows += [f"{symbol} {x:.12f} {y:.12f} {z:.12f}"
             for symbol, (x, y, z) in zip(specification["symbols"], specification["coordinates"])]
    rows += ["units angstrom", "no_com", "no_reorient", "symmetry c1"]
    molecule = psi4.geometry("\n".join(rows))
    psi4.core.clean_options()
    psi4.set_options({
        "basis": BASIS, "reference": "rhf" if mult == 1 else "uhf", "scf_type": "pk",
        "e_convergence": 1e-10, "d_convergence": 1e-9, "maxiter": 80, "fail_on_maxiter": True,
    })
    start = perf_counter()
    energy, wfn = psi4.energy("scf", molecule=molecule, return_wfn=True)
    seconds = perf_counter() - start
    assert np.isfinite(energy)
    assert (wfn.nalpha(), wfn.nbeta()) == expected_counts
    assert np.allclose(np.asarray(molecule.geometry()) * psi4.constants.bohr2angstroms,
                       specification["coordinates"], atol=1e-9)
    return {"energy": energy, "wfn": wfn, "seconds": seconds, "electrons": electrons}

results = {}
for name in ("OH+", "OH", "OH-"):
    results[name] = single_point(specifications[name])
    r = results[name]
    print(f"{name:4s}: {r['energy']:.10f} Eh; {r['electrons']} electrons; {r['seconds']:.3f} s")

In [ ]:
# The two additional fragments complete the balanced reactions below.
for name in ("H2O", "H"):
    results[name] = single_point(specifications[name])
    r = results[name]
    print(f"{name:4s}: {r['energy']:.10f} Eh; {r['electrons']} electrons; {r['seconds']:.3f} s")
assert results["H"]["energy"] >= -0.5 - 1e-8  # exact nonrelativistic H limit
print("The isolated H atom provides an independent variational sanity check.")

## 8.2.3 Spin diagnostics and density matrices

A pure total-spin state has $\langle\hat S^2\rangle=S(S+1)$ in units of $\hbar^2$: 0 for a singlet, 0.75 for a doublet, and 2 for a triplet. A UHF determinant is an eigenfunction of $\hat S_z$, but generally not of $\hat S^2$.

For occupied alpha and beta coefficient matrices $C^\alpha_{\mathrm{occ}}$, $C^\beta_{\mathrm{occ}}$, with AO overlap matrix $S_{\mathrm{AO}}$, define

$$O_{\alpha\beta}=(C^\alpha_{\mathrm{occ}})^T S_{\mathrm{AO}} C^\beta_{\mathrm{occ}},$$
$$\langle\hat S^2\rangle=M_S(M_S+1)+N_\beta-\sum_{ij}|O_{\alpha\beta,ij}|^2,$$

for $N_\alpha\ge N_\beta$. The following computes the diagnostic from the actual orbitals rather than assuming the input multiplicity guarantees spin purity. The density checks are $\mathrm{Tr}[(D^\alpha+D^\beta)S_{\mathrm{AO}}]=N$ and $\mathrm{Tr}[(D^\alpha-D^\beta)S_{\mathrm{AO}}]=N_\alpha-N_\beta$.

A small spin deviation is useful evidence about a determinant, but it does not prove the correct electronic configuration, orbital stability, or accurate energy. There is no universal spin-deviation cutoff that certifies every chemical calculation.

In [ ]:
def density_and_spin_checks(wfn, multiplicity):
    overlap = np.asarray(psi4.core.MintsHelper(wfn.basisset()).ao_overlap())
    da, db = np.asarray(wfn.Da()), np.asarray(wfn.Db())
    ca = np.asarray(wfn.Ca())[:, :wfn.nalpha()]
    cb = np.asarray(wfn.Cb())[:, :wfn.nbeta()]
    assert np.allclose(ca.T @ overlap @ ca, np.eye(wfn.nalpha()), atol=1e-8)
    assert np.allclose(cb.T @ overlap @ cb, np.eye(wfn.nbeta()), atol=1e-8)
    electrons = np.trace((da + db) @ overlap)
    spin_excess = np.trace((da - db) @ overlap)
    assert np.isclose(electrons, wfn.nalpha() + wfn.nbeta(), atol=1e-8)
    assert np.isclose(spin_excess, wfn.nalpha() - wfn.nbeta(), atol=1e-8)
    ms = (wfn.nalpha() - wfn.nbeta()) / 2
    s_squared = ms * (ms + 1) + wfn.nbeta() - np.sum(np.abs(ca.T @ overlap @ cb)**2)
    target_s = (multiplicity - 1) / 2
    target = target_s * (target_s + 1)
    assert s_squared >= target - 1e-8
    return {"electron_trace": float(electrons), "spin_trace": float(spin_excess),
            "s_squared": float(s_squared), "target": target, "excess": float(s_squared - target)}

print(f"{'Species':7s} {'N':>6s} {'Nα−Nβ':>7s} {'<S²>':>10s} {'Target':>8s} {'Excess':>10s}")
for name, result in results.items():
    result["spin"] = density_and_spin_checks(result["wfn"], specifications[name]["multiplicity"])
    s = result["spin"]
    print(f"{name:7s} {s['electron_trace']:6.2f} {s['spin_trace']:7.2f} {s['s_squared']:10.6f} {s['target']:8.2f} {s['excess']:10.6f}")
assert np.isclose(results["H"]["spin"]["s_squared"], 0.75, atol=1e-8)
assert abs(results["OH-"]["spin"]["s_squared"]) < 1e-8

In [ ]:
names = ["OH+", "OH", "OH-", "H2O", "H"]
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5), layout="constrained")
axes[0].bar(names, [results[n]["spin"]["spin_trace"] for n in names], color="steelblue")
axes[0].set(ylabel="Integrated spin density (Nα − Nβ)", title="Spin population excess")
axes[1].bar(names, [results[n]["spin"]["excess"] for n in names], color="darkorange")
axes[1].axhline(0, color="black", linewidth=0.7)
axes[1].set(ylabel=r"$\langle S^2\rangle - S(S+1)$ ($\hbar^2$)", title="UHF spin contamination diagnostic")
fig.savefig(OUT / "spin_diagnostics.png", dpi=140)
plt.show()

The integral of spin density is a spin-population excess, not a general count of all unpaired electrons. An open-shell singlet can have zero integrated spin density. RHF's zero spin density does not prove that a closed-shell determinant is an adequate model of every singlet.

## 8.2.4 Cation formation and electron attachment

Write the electron explicitly. For a common geometry $\mathbf R$, and a separated free electron assigned zero electronic energy,

$$\mathrm{OH}\rightarrow\mathrm{OH}^+ + e^-:
\quad \Delta E_{\mathrm{ion}}=E_{\mathrm{OH}^+}(\mathbf R)-E_{\mathrm{OH}}(\mathbf R),$$
$$\mathrm{OH}+e^-\rightarrow\mathrm{OH}^-:
\quad \Delta E_{\mathrm{attach}}=E_{\mathrm{OH}^-}(\mathbf R)-E_{\mathrm{OH}}(\mathbf R).$$

The electron affinity uses the **energy released** convention: $\mathrm{EA}=-\Delta E_{\mathrm{attach}}$. A positive EA favors attachment. These are **fixed-geometry, state-specific electronic** differences; our input is not an optimized neutral minimum. Adiabatic comparisons allow the species to relax, with zero-point corrections needed to compare vibrational ground-state thresholds. See the IUPAC definitions of [ionization energy](https://doi.org/10.1351/goldbook.I03199) and [electron affinity](https://doi.org/10.1351/goldbook.E01977).

We use a small balance checker so that forgetting an electron or proton produces an error rather than an unexplained energy difference.

In [ ]:
# Stoichiometric coefficients are positive for products, negative for reactants.
composition = {name: {element: spec["symbols"].count(element) for element in ("H", "O")}
               for name, spec in specifications.items()}
charge = {name: spec["charge"] for name, spec in specifications.items()}
energies = {name: result["energy"] for name, result in results.items()}
electrons = {name: result["electrons"] for name, result in results.items()}
# Nonrelativistic electronic-energy references: isolated proton and free electron at rest.
composition.update({"H+": {"H": 1, "O": 0}, "e-": {"H": 0, "O": 0}})
charge.update({"H+": 1, "e-": -1})
electrons.update({"H+": 0, "e-": 1})
energies.update({"H+": 0.0, "e-": 0.0})

def reaction_energy(stoichiometry):
    for element in ("H", "O"):
        if sum(nu * composition[name][element] for name, nu in stoichiometry.items()) != 0:
            raise ValueError(f"Reaction does not conserve {element}.")
    if sum(nu * charge[name] for name, nu in stoichiometry.items()) != 0:
        raise ValueError("Reaction does not conserve charge.")
    assert sum(nu * electrons[name] for name, nu in stoichiometry.items()) == 0
    return sum(nu * energies[name] for name, nu in stoichiometry.items())

ionization = reaction_energy({"OH": -1, "OH+": 1, "e-": 1})
attachment = reaction_energy({"OH": -1, "e-": -1, "OH-": 1})
electron_affinity = -attachment
print(f"OH → OH+ + e−: ΔE = {ionization * HARTREE_TO_EV:.4f} eV")
print(f"OH + e− → OH−: ΔE = {attachment * HARTREE_TO_EV:.4f} eV")
print(f"EA = −ΔEattach = {electron_affinity * HARTREE_TO_EV:.4f} eV")
try:
    reaction_energy({"OH": -1, "OH+": 1})
except ValueError as error:
    print("Self-check (electron omitted):", error)
else:
    raise AssertionError("Unbalanced ionization was accepted.")

**A scientifically useful failure:** with these settings, HF gives a negative electron affinity for OH. That does not establish that real hydroxide is unbound. [NIST lists experimental OH electron affinities near +1.8277 eV](https://webbook.nist.gov/cgi/cbook.cgi?ID=C3352576&Mask=460&Units=CAL). Those are threshold measurements, whereas our number is a fixed-geometry electronic estimate; they are not identical observables. Nonetheless, the sign failure shows that this small, uncorrelated calculation must not be used as a reliable attachment prediction.

Diffuse functions are necessary for many anions but are not a guarantee of accuracy or electron binding. A finite Gaussian basis can represent an apparently localized electronic solution even when electron detachment is energetically favorable in the model. Electronic correlation, basis convergence, geometry, and the state being compared need examination. SCF convergence alone does not resolve those issues.

## 8.2.5 Anions, proton loss, and radical bond cleavage

For the same water starting structure, distinguish two different reactions:

$$\mathrm{H_2O}\rightarrow\mathrm{OH}^-+\mathrm H^+\quad\text{(heterolytic proton loss)},$$
$$\mathrm{H_2O}\rightarrow\mathrm{OH}^{\bullet}+\mathrm H^{\bullet}\quad\text{(homolytic cleavage)}.$$

A proton contains no electrons, so its nonrelativistic **electronic** energy is zero. It does not have zero translational entropy or zero solvation free energy. The remaining OH fragment is held at the original bond length; these are separated-fragment electronic cleavage costs, not optimized bond dissociation enthalpies or activation barriers. Two infinitely separated doublet radicals may couple to singlet or triplet total spin; summing their isolated energies does not calculate a reaction path.

Solution acidity instead concerns a standard free energy. For a dimensionless acid-dissociation equilibrium constant, $pK_a=\Delta G^\circ/(RT\ln 10)$. It requires consistent solution standard states, solvent contributions, thermal terms, and a proton/reference convention. Applying that equation directly to our gas-phase electronic difference would be incorrect. [NIST's ion-thermochemistry guide](https://webbook.nist.gov/chemistry/ion/) describes gas-phase acidity and proton-transfer measurements.

In [ ]:
deprotonation = reaction_energy({"H2O": -1, "OH-": 1, "H+": 1})
homolysis = reaction_energy({"H2O": -1, "OH": 1, "H": 1})
print(f"H2O → OH− + H+: ΔE = {deprotonation * HARTREE_TO_KJ_MOL:.2f} kJ/mol")
print(f"H2O → OH• + H•: ΔE = {homolysis * HARTREE_TO_KJ_MOL:.2f} kJ/mol")

# Hess's-law identity using exactly the same computed species and conventions:
# heterolysis − homolysis = IE(H) − EA(OH).
hydrogen_ionization = reaction_energy({"H": -1, "H+": 1, "e-": 1})
assert np.isclose(deprotonation - homolysis, hydrogen_ionization - electron_affinity, atol=1e-10)
print("Energy-cycle check passed: ΔEheterolysis − ΔEhomolysis = IE(H) − EA(OH).")

In [ ]:
labels = ["OH → OH⁺ + e⁻", "OH + e⁻ → OH⁻", "H₂O → OH⁻ + H⁺", "H₂O → OH• + H•"]
values = np.array([ionization, attachment, deprotonation, homolysis]) * HARTREE_TO_KJ_MOL
fig, ax = plt.subplots(figsize=(8.4, 4.1), layout="constrained")
bars = ax.barh(labels, values, color=["teal", "steelblue", "darkorange", "purple"])
ax.bar_label(bars, fmt="%.1f", padding=4)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="Products − reactants electronic energy (kJ/mol)",
       title="Different balanced gas-phase processes at specified geometries")
ax.set_xlim(min(-40, values.min()*1.1), values.max()*1.17)
ax.invert_yaxis()
fig.savefig(OUT / "balanced_reactions.png", dpi=140)
plt.show()

### Worked analysis: close an energy cycle before interpreting it

Start from water and reach the **same final fragments**, $\mathrm{OH^-+H^+}$, by two bookkeeping routes. One directly uses the heterolysis energy. The other first makes neutral radicals, ionizes H, and attaches the released electron to OH:

$$\Delta E_{\rm heterolysis}=\Delta E_{\rm homolysis}+\mathrm{IE(H)}-\mathrm{EA(OH)}.$$

The free electron uses the same zero-energy convention throughout. Each horizontal level below is a sum of previously computed fragment energies at the stated geometries. Arrows show additions and subtractions of energy; **they are not a proposed physical mechanism or activation barriers**.

In [ ]:
cycle_levels = np.array([0.0, homolysis, homolysis + hydrogen_ionization,
                         deprotonation]) * HARTREE_TO_KJ_MOL
cycle_labels = ["H₂O", "OH• + H•", "OH• + H⁺ + e⁻", "OH⁻ + H⁺"]
cycle_steps = [homolysis, hydrogen_ionization, -electron_affinity]
np.testing.assert_allclose(np.diff(cycle_levels),
                           np.asarray(cycle_steps) * HARTREE_TO_KJ_MOL, atol=1e-7)
fig, ax = plt.subplots(figsize=(9, 4.8), layout="constrained")
for i, (level, label) in enumerate(zip(cycle_levels, cycle_labels)):
    ax.hlines(level, i - 0.23, i + 0.23, lw=3, color="#28788e")
    ax.text(i, level + 42, label, ha="center", va="bottom", fontsize=10)
    if i:
        ax.annotate("", (i - 0.26, level), (i - 0.74, cycle_levels[i-1]),
                    arrowprops={"arrowstyle": "->", "color": "0.4"})
        ax.text(i - 0.5, (level + cycle_levels[i-1])/2 - 70,
                f"{cycle_steps[i-1] * HARTREE_TO_KJ_MOL:+.1f}", ha="center", fontsize=9)
ax.set(xticks=[], xlabel="Energy accounting route; not a reaction coordinate",
       ylabel="Electronic energy relative to H₂O (kJ/mol)",
       title="Same endpoints, same total energy change",
       xlim=(-0.4, 3.45), ylim=(-140, cycle_levels.max()+210))
fig.savefig(OUT / "fragment_energy_cycle.png", dpi=140)
plt.show()
print(f"Cycle closure error: {(cycle_levels[-1] - sum(cycle_steps)*HARTREE_TO_KJ_MOL):.2e} kJ/mol")

**Conclusion.** The cycle closes because each species uses one consistent energy. This detects sign, unit, and fragment-accounting errors. It does **not** establish agreement with experiment: the HF electron-affinity limitation discussed above remains even when closure is exact.

**Next research step:** compare like-for-like observables with a suitable higher-level method, geometry treatment, and reference data. For solution acidity, add a thermodynamic treatment of solvation and standard states; the gas-phase electronic cycle alone is not a pKa calculation.

## 8.2.6 Applying the bookkeeping to organic intermediates

Comparing methyl, ethyl, tert-butyl, or benzylic species requires a reference that balances the different formulas. A raw bar chart of their total energies is not a substituent-stability scale. Distinguish **alkyl**, **allylic** (adjacent to C=C), and **benzylic** (adjacent to an aromatic ring) structures before discussing stabilization.

Useful reference reactions include:

| Question | Balanced comparison | Meaning of products minus reactants |
| --- | --- | --- |
| Relative carbocation formation | $\mathrm{R-H+CH_3^+\rightarrow R^++CH_4}$ | A hydride-transfer reaction energy; compare identical computational conventions |
| Relative radical formation | $\mathrm{R-H+CH_3^{\bullet}\rightarrow R^{\bullet}+CH_4}$ | Difference between electronic C–H homolysis costs using methane as reference |
| Relative acidity | $\mathrm{AH+B^-\rightarrow A^-+BH}$ | Proton-transfer energy; a solution $\Delta G^\circ$ gives a relative $pK_a$ |

Thus, a difference of parent/substituted energy differences can be meaningful **when its complete reference reaction is stated**. It is not automatically an absolute radical stability, ionization energy, or aqueous acidity. Neutral radical precursors lose a hydrogen atom in homolysis; neutral molecules lose an electron in ionization. Those processes must not be confused.

No larger organic calculation is required here. The balance checker and the energy-cycle identity provide a compact way to audit such future comparisons.

In [ ]:
record = {
    "method": "HF/6-31+G*; RHF singlets, UHF doublets/triplet", "psi4": psi4.__version__,
    "oh_distance_angstrom": bond, "water_angle_degrees": 104.5,
    "species": {name: {"energy_hartree": r["energy"], "seconds": r["seconds"], "spin": r["spin"],
                         "charge": specifications[name]["charge"],
                         "multiplicity": specifications[name]["multiplicity"],
                         "coordinates_angstrom": specifications[name]["coordinates"].tolist()}
                for name, r in results.items()},
    "reaction_energies_hartree": {"ionization": ionization, "attachment": attachment,
                                  "deprotonation": deprotonation, "homolysis": homolysis},
    "interpretation": "Specified fixed geometries; electronic energies only; no optimization or thermochemistry",
}
(OUT / "results.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
psi4.core.clean()
psi4.core.clean_options()
print(f"Saved checked results, log, and figures to {OUT}")

## Exercises and selected answers

1. How many alpha and beta electrons are in the chosen OH⁺ state? Why is assigning multiplicity 1 to every cation incorrect?
2. Write a balanced electron-detachment reaction for OH⁻. Derive its fixed-geometry energy from the attachment calculation.
3. Why is $E(\mathrm{OH})-E(\mathrm{H_2O})$ alone not a homolytic O–H bond dissociation energy?
4. If a UHF doublet gives $\langle S^2\rangle=1.20$, what is its excess over the target? Does SCF convergence remove it?
5. Use the computed numbers to verify the Hess's-law identity on paper. Explain why a wrong sign for EA does not stop the algebraic identity from holding.
6. A proton-transfer solution free energy for $\mathrm{AH+B^-\to A^-+BH}$ is negative. Which acid has the smaller $pK_a$, assuming a common solvent and standard-state convention?

<details><summary>Selected answers</summary>

1. $N_\alpha=5$, $N_\beta=3$. Charge fixes electron number, not how spins couple; the chosen cation is an even-electron triplet.
2. $\mathrm{OH^-\to OH+e^-}$. Its energy is $-\Delta E_{\mathrm{attach}}=\mathrm{EA}$ at the same geometry.
3. A hydrogen atom is missing. Homolysis requires $E(\mathrm{OH})+E(\mathrm H)-E(\mathrm{H_2O})$, followed by geometry and thermal corrections if an enthalpy is wanted.
4. The excess is $1.20-0.75=0.45$. A converged UHF determinant may remain spin contaminated.
5. Hess's law checks consistent energy bookkeeping; it cannot establish that the underlying model is accurate.
6. AH: $\Delta G^\circ=RT\ln 10\,[pK_a(\mathrm{AH})-pK_a(\mathrm{BH})]$ for this proton-transfer reaction.

</details>

Continue with [Part 3: molecular orbitals](Chapter08_Part3.ipynb).